In [1]:
#This code takes the original data, the prediction of nns and the BMS, computes the rmse and mae and saves everything into a dataframe 

In [8]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

In [9]:
def clean_index(dataframe):
    dataframe.set_index('Unnamed: 0', inplace=True)
    dataframe.index.name = None
    dataframe= dataframe.reset_index(drop=True)
    return dataframe

def add_bms_pred(dataframe, bms_trace, number_param):
    VARS = ['x1',]
    x = dn[[c for c in VARS]].copy()
    y=dataframe.noise

    if number_param==10:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np10.2017-10-18 18:07:35.089658.dat')
    elif number_param==20:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np20.maxs200.2024-05-10 162907.551306.dat')

    #mdl model
    minrow = bms_trace[bms_trace.H == min(bms_trace.H)].iloc[0]
    minH, minexpr, minparvals = minrow.H, minrow.expr, ast.literal_eval(minrow.parvals)
    t = Tree(
        variables=list(x.columns),
        parameters=['a%d' % i for i in range(number_param)],
        x=x, y=y,
        prior_par=prior_par,
        max_size=200,
        from_string=minexpr,
    )

    t.set_par_values(deepcopy(minparvals))

    dplot = deepcopy(dn)
    dplot['ybms'] = t.predict(x)

    return dplot
    

In [10]:
#Read NN and BMS data
functions=['leaky_ReLU', 'tanh'] #tanh, leaky_ReLU
realizations=2
N=9

sigmas=[0.0, 0.02, 0.04,0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20]
resolution='0.5x' #0.5x, 1x, 2x, 4e-3x
resolutions={'0.5x':'0.1', '1x':'0.05' , '2x': '0.025' , '4e-3x':'0.004' }

NPAR=10 #10, 20
steps=50000



rmse_nn_train=[];rmse_nn_test=[]
rmse_mdl_train=[];rmse_mdl_test=[]

mae_nn_train=[];mae_nn_test=[]
mae_mdl_train=[];mae_mdl_test=[]

n_index=[];r_index=[];sigma_index=[];function_index=[]

#Put mae and rmse of each simulation (on nn and bms) in a dataframe
for function in functions:

    for sigma in sigmas:

        for r in range(realizations+1):

            file_model='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            model_d='../../data/nns/' + resolution + '_resolution/approximation/' + file_model
            d=pd.read_csv(model_d)

            n_points=int(len(d.index)/10)
            train_fraction=3/4;train_size=int(n_points*train_fraction)
            

            for n in range(N+1):
                n_index.append(n);r_index.append(r);sigma_index.append(sigma);function_index.append(function)
            
                dn=d[d['rep']==n]
                dn=clean_index(dn)

                #Read BMS data
                filename='BMS_'+function+'_n_'+str(n)+'_sigma_'+str(sigma)+ '_r_' + str(r) + '_res_' + resolutions[resolution] + '_trace_'\
                +str(steps)+'_prior_'+str(NPAR)+ '.csv'

                print(function, sigma, n, r)
                
                try:
                    print("hello")
                    trace=pd.read_csv('../../data/MSTraces/' + resolution + '_resolution/' + filename, sep=';', header=None,
                                      names=['t','H','expr','parvals','kk1','kk2','kk3'])
                    print("bye")
                    dplot=add_bms_pred(dn, trace, NPAR)
                except FileNotFoundError:
                    dplot = deepcopy(dn) #If no bms errors available, fill the dataframe with zeros
                    dplot['ybms'] = [0] * len(dplot)
                

                #Compute errors
                #-----------------------------------------------------------------------------------------------------------------------
                #nns
                rmse_nn_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                rmse_nn_train.append(rmse_nn_train_i)
            
                rmse_nn_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                rmse_nn_test.append(rmse_nn_test_i)

                mae_nn_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                mae_nn_train.append(mae_nn_train_i)
            
                mae_nn_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                mae_nn_test.append(mae_nn_test_i)

                
                #bms
                try:
                    rmse_mdl_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ybms'],dn.loc[:train_size-1]['y'])
                except ValueError:
                    rmse_mdl_train_i=np.inf
                rmse_mdl_train.append(rmse_mdl_train_i)

                try:
                    rmse_mdl_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ybms'],dn.loc[train_size-1:]['y'])
                except (ValueError, RuntimeWarning) as e:
                    rmse_mdl_test_i=np.inf
                
                rmse_mdl_test.append(rmse_mdl_test_i)

                try:
                    mae_mdl_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ybms'],dplot.loc[:train_size -1]['y'])
                except ValueError:
                    mae_mdl_train_i=np.inf
                mae_mdl_train.append(mae_mdl_train_i)

                try:
                    mae_mdl_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ybms'],dplot.loc[train_size -1:]['y'])
                except ValueError:
                    mae_mdl_test_i=np.inf
                
                mae_mdl_test.append(mae_mdl_test_i)
                #-----------------------------------------------------------------------------------------------------------------------


#Save all in a dataframe
errors_df=pd.DataFrame({'sigma':sigma_index, 'function':function_index, 'mae_nn_train':mae_nn_train, 'mae_nn_test':mae_nn_test, 'mae_mdl_train':mae_mdl_train, 
                        'mae_mdl_test':mae_mdl_test, 'rmse_nn_train':rmse_nn_train, 'rmse_nn_test': rmse_nn_test, 
                        'rmse_mdl_train':rmse_mdl_train, 'rmse_mdl_test': rmse_mdl_test, 'n':n_index, 'r': r_index})
errors_df.to_csv('../../data/test_errors_median_' + resolution + '.csv')
display(errors_df)

leaky_ReLU 0.0 0 0
hello
bye
leaky_ReLU 0.0 1 0
hello


<lambdifygenerated-23105>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*x1**x1 + x1
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-23106>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*x1**x1 + x1
<lambdifygenerated-23107>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*_a6_**x1 + x1
<lambdifygenerated-23109>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*_a6_**(x1**x1) + x1
<lambdifygenerated-23110>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*_a6_**(x1**x1) + x1
<lambdifygenerated-23113>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*_a6_**(_a3_**sinh(x1)) + x1
<lambdifygenerated-23115>:2: RuntimeWarning: invalid value encountered in power
  return _a1_*_a6_**(_a3_**sinh(x1**2)) + x1
/usr/local/l

bye
leaky_ReLU 0.0 2 0
hello
bye
leaky_ReLU 0.0 3 0
hello
bye
leaky_ReLU 0.0 4 0
hello


<lambdifygenerated-23161>:2: RuntimeWarning: invalid value encountered in power
  return x1**x1
<lambdifygenerated-23162>:2: RuntimeWarning: invalid value encountered in power
  return x1**x1
<lambdifygenerated-23165>:2: RuntimeWarning: invalid value encountered in power
  return _a2_**(x1**x1)
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-23166>:2: RuntimeWarning: invalid value encountered in power
  return _a2_**(x1**x1)
<lambdifygenerated-23167>:2: RuntimeWarning: invalid value encountered in power
  return _a2_**(_a1_**x1)
/usr/local/lib/python3.10/dist-packages/scipy/optimize/_minpack_py.py:497: RuntimeWarning: overflow encountered in matmul
  cov_x = invR @ invR.T
/usr/local/lib/python3.10/dist-packages/scipy/optimize/_minpack_py.py:1055: RuntimeWarning: invalid value encountered in multiply
  pcov = pcov * s_sq
<lam

bye
leaky_ReLU 0.0 5 0
hello
bye
leaky_ReLU 0.0 6 0
hello
bye
leaky_ReLU 0.0 7 0
hello
bye
leaky_ReLU 0.0 8 0
hello


...

/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(


bye
tanh 0.2 2 2
hello
bye
tanh 0.2 3 2
hello
bye
tanh 0.2 4 2
hello
bye
tanh 0.2 5 2
hello
bye
tanh 0.2 6 2
hello
bye
tanh 0.2 7 2
hello
bye
tanh 0.2 8 2
hello
bye


<lambdifygenerated-34597>:2: RuntimeWarning: invalid value encountered in power
  return exp(_a2_*x1**x1)
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-34598>:2: RuntimeWarning: invalid value encountered in power
  return exp(_a2_*x1**x1)


tanh 0.2 9 2
hello
bye


,sigma,function,mae_nn_train,mae_nn_test,mae_mdl_train,mae_mdl_test,rmse_nn_train,rmse_nn_test,rmse_mdl_train,rmse_mdl_test,n,r
0,0.0,leaky_ReLU,0.010042,0.116551,0.001032,0.063266,0.011877,0.133142,0.001275,0.082951,0,0
1,0.0,leaky_ReLU,0.006217,0.169584,0.023344,0.269050,0.008977,0.192922,0.028029,0.296079,1,0
2,0.0,leaky_ReLU,0.008492,0.105571,0.014257,0.086586,0.011984,0.137861,0.016435,0.110833,2,0
3,0.0,leaky_ReLU,0.004126,0.085344,0.004799,0.203640,0.005523,0.103214,0.006336,0.246610,3,0
4,0.0,leaky_ReLU,0.004736,0.044408,0.003880,0.506176,0.006189,0.051462,0.004796,0.726888,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,tanh,0.126637,0.723464,0.060303,0.831309,0.161518,0.886390,0.106253,0.890840,5,2
656,0.2,tanh,0.094209,0.467383,0.044108,0.263869,0.136488,0.514125,0.066985,0.266259,6,2
657,0.2,tanh,0.138680,1.730582,0.094496,0.012186,0.175718,2.154303,0.113941,0.017034,7,2
658,0.2,tanh,0.168881,2.293381,0.163269,0.748055,0.201022,2.718679,0.224000,0.750894,8,2
